In [22]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import joblib


In [23]:
DATA_PATH = "../data/customer_churn_data.csv"
OUTPUT_DIR = "../data_processed"
RANDOM_STATE = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.2
gap = 5

In [24]:
# load data
df = pd.read_csv(DATA_PATH)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [25]:
le = LabelEncoder()
scaler = StandardScaler()

In [26]:
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()


In [27]:
# encoders = {}
# for col in categorical_cols:
#     le = LabelEncoder()
#     df[col] = le.fit_transform(df[col].astype(str))
#     encoders[col] = le

In [28]:
TEST_SIZE = 0.2
VAL_SIZE = 0.2
gap = 5
n_train = int(len(df) * (1 - TEST_SIZE - VAL_SIZE)) - 2* gap
n_val = int(len(df) * VAL_SIZE)
n_test = len(df) - n_train - n_val + gap

df_train = df.iloc[:n_train]
df_val = df.iloc[n_train+gap:n_train +gap+ n_val]
df_test = df.iloc[n_train +gap + n_val:]

In [29]:
df_train.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [30]:
def encode_features(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Encode categorical features using LabelEncoder."""
    df = df.copy()
    encoders = {}

    categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

    # Remove target from categorical columns if present
    # if "Churn" in categorical_cols:
    #     categorical_cols.remove("Churn")

    for col in categorical_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        encoders[col] = le

    return df, encoders

In [31]:
df_train, train_encoders = encode_features(df_train)
df_val, val_encoders = encode_features(df_val)  
df_test, test_encoders = encode_features(df_test)

In [32]:
df_train.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,3221,0,0,1,0,1,0,1,0,0,...,0,0,0,0,0,1,2,29.85,1551,0
1,2358,1,0,0,0,34,1,0,0,2,...,2,0,0,0,1,0,3,56.95,900,0
2,1509,1,0,0,0,2,1,0,0,2,...,0,0,0,0,0,1,3,53.85,94,1
3,3323,1,0,0,0,45,0,1,0,2,...,2,2,0,0,1,0,0,42.30,853,0
4,3904,0,0,0,0,2,1,0,1,0,...,0,0,0,0,0,1,2,70.70,553,1


In [33]:
X_train, y_train = df_train.drop("Churn", axis=1), df_train["Churn"]
X_val, y_val = df_val.drop("Churn", axis=1), df_val["Churn"]
X_test, y_test = df_test.drop("Churn", axis=1), df_test["Churn"]

In [34]:
X_train.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,3221,0,0,1,0,1,0,1,0,0,2,0,0,0,0,0,1,2,29.85,1551
1,2358,1,0,0,0,34,1,0,0,2,0,2,0,0,0,1,0,3,56.95,900
2,1509,1,0,0,0,2,1,0,0,2,2,0,0,0,0,0,1,3,53.85,94
3,3323,1,0,0,0,45,0,1,0,2,0,2,2,0,0,1,0,0,42.30,853
4,3904,0,0,0,0,2,1,0,1,0,0,0,0,0,0,0,1,2,70.70,553


In [35]:
def scale_features(X: pd.DataFrame, scaler: StandardScaler = None) -> tuple[pd.DataFrame, StandardScaler]:
    """Scale numerical features using StandardScaler."""
    numerical_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

    if scaler is None:
        scaler = StandardScaler()
        X[numerical_cols] = scaler.fit_transform(X[numerical_cols])
    else:
        X[numerical_cols] = scaler.transform(X[numerical_cols])

    return X, scaler

In [36]:
df_train = df_train.drop(columns=["customerID"])
df_val = df_val.drop(columns=["customerID"])
df_test = df_test.drop(columns=["customerID"])

In [37]:
X_train, scaler = scale_features(df_train.drop(columns=["Churn"]))
X_val, _ = scale_features(df_val.drop(columns=["Churn"]), scaler)
X_test, _ = scale_features(df_test.drop(columns=["Churn"]), scaler)

In [38]:
X_train.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,0,0,1,0,-1.265160,0,1,0,0,2,0,0,0,0,0,1,2,-1.160879,-0.383064
1,1,0,0,0,0.075467,1,0,0,2,0,2,0,0,0,1,0,3,-0.260223,-0.944554
2,1,0,0,0,-1.224535,1,0,0,2,2,0,0,0,0,0,1,3,-0.363250,-1.639733
3,1,0,0,0,0.522343,0,1,0,2,0,2,2,0,0,1,0,0,-0.747109,-0.985092
4,0,0,0,0,-1.224535,1,0,1,0,0,0,0,0,0,0,1,2,0.196752,-1.243843
